In [ ]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
!pip install pyarrow==8.0.0  # Downgrade pyarrow to a known compatible version
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset, Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 27.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 62.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 16.1.0 which

In [ ]:

# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'google/muril-base-cased'  # Changed to MuRIL model
tokenizer_name_or_path = 'google/muril-base-cased'  # Changed to MuRIL model
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-5
num_epochs = 50
batch_size = 8


In [ ]:
# Step 3: Load Dataset

# dataset = pd.read_csv()

# dataset.iloc[0]

column_names = [label_column, "blank", text_column]
# Load the dataset without headers and assign the column names
dataset = pd.read_csv("/content/drive/MyDrive/societal_2l_train_hi.csv", header=None, names=column_names).head(500) # examples in dataset

dataset.drop("blank", axis=1, inplace=True)
dataset[label_column] = dataset[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset.head())
# Extract the specific range
# specific_range_dataset = dataset.iloc[1000:1201]  # Adjusted to include index 1200
# print(specific_range_dataset.head())
from sklearn.model_selection import train_test_split

# Split the preprocessed dataset into training and testing sets (80% train, 20% test)
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Now you have train_dataset and test_dataset for training and testing respectively
print(train_dataset)
dataset_train = Dataset.from_pandas(train_dataset)
dataset_test = Dataset.from_pandas(test_dataset)

   text_label                                         Tweet text
0           0  @sardanarohit @sudhirchaudhary @arnabgoswamias...
1           1  @narendramodi प्रिय पीएम, युवाओं को हम #Cashle...
2           0  @Narendramodi सर पठानकोट निगम AMRIT CITY SCHEO...
3           1  #jobs #jobsearch ##parliament GST बिल पास करता...
4           1  @Upsubramanya ppl praise manmohan 4 1991 सुधार...
     text_label                                         Tweet text
249           1  #GST को अच्छी तरह से समझें और इसे अपने निर्वाच...
433           1  @YADAVAKHILESH आप असली SAMAJWADI हैं। ऐसे व्यक...
19            0  यदि पाक के लोग आतंकवाद के खिलाफ खड़े होने के ल...
322           0  Pathankot के दागी SP SALWINDER ने @Timesofindi...
332           0  @Pmoindia, @finminindia @narendramodi केंद्र स...
..          ...                                                ...
106           0  RT @kumar_ke5hav: #surgicalStrike और एक कमजोर ...
270           0  @airtelindia @ideacellular @vodafonein @aircel...
348    

In [ ]:
column_names = [label_column,"blank" ,text_column]
# Load the dataset without headers and assign the column names


#1% - 52, 2%- 104, 10% - 520
number = 520
dataset_train = pd.read_csv("/content/drive/MyDrive/train/train_dataset_Urdu.csv", header=None, names=column_names).head(number)

dataset_test = pd.read_csv("/content/drive/MyDrive/test/test_dataset_Urdu.csv", header=None, names=column_names)

dataset_train.drop("blank", axis=1, inplace=True)
dataset_train[label_column] = dataset_train[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset_train.head())

dataset_test.drop("blank", axis=1, inplace=True)
dataset_test[label_column] = dataset_test[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset_test.head())

   text_label                                         Tweet text
0           0  @aajtak @abpnewshindi تمام مخالف بھارت کے لیے ...
1           0  @سوامی 39 صرف اس وجہ سے کہ آپ رکن پارلیمنٹ ہیں...
2           0  bdutt sagarikaghose آپ کو یہ جاننا ہوگا کہ آپ ...
3           1  آسام اسمبلی نے متفقہ طور پر جی ایس ٹی پر آئین ...
4           0  rt @mnomics_: gst منسوخ کرن مین 5 سیل Bitaaaya...
   text_label                                         Tweet text
0           0  @aajtak @abpnewshindi تمام مخالف بھارت کے لیے ...
1           0  timesnow u ماؤں fuckers .. ہم موت کے بعد نہیں ...
2           1  @نارینڈرموڈی عظیم اقدامات اٹھاتے رہیں ، جان لی...
3           1  #TOI #NewsINDIA حکومت کو جی ایس ٹی جمع کرنے کا...
4           1  #URI 28 ، #برون 13 یہاں کنگسٹن میں۔2010 کے بعد...


In [ ]:
# Now you have train_dataset and test_dataset for training and testing respectively
from datasets import Dataset
dataset_train = Dataset.from_pandas(dataset_train)
dataset_test = Dataset.from_pandas(dataset_test)

In [ ]:
# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs


In [ ]:
# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)


Running tokenizer on dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)


In [ ]:
# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())
# Add dropout to prevent overfitting
model.dropout = torch.nn.Dropout(0.1)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,682 || all params: 237,565,444 || trainable%: 0.0032
None


In [ ]:
# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

In [ ]:
# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()


In [ ]:

    model.eval()
    eval_loss = 0
    eval_preds = []
    true_labels = []
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
        true_labels.extend(batch["labels"].detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
    accuracy = accuracy_score(true_labels, eval_preds)
    f1 = f1_score(true_labels, eval_preds, average='weighted')
    recall = recall_score(true_labels, eval_preds, average='weighted')
    precision = precision_score(true_labels, eval_preds, average='weighted')

    print(f"Epoch {epoch}: train_loss={train_epoch_loss}, eval_loss={eval_epoch_loss}, accuracy={accuracy}, f1_score={f1}, recall={recall}, precision={precision}")

100%|██████████| 13/13 [00:00<00:00, 13.96it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 0: train_loss=0.6932167410850525, eval_loss=0.6930893063545227, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25


100%|██████████| 13/13 [00:00<00:00, 14.26it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1: train_loss=0.6931026577949524, eval_loss=0.6930925250053406, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25


100%|██████████| 13/13 [00:00<00:00, 14.29it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

100%|██████████| 13/13 [00:00<00:00, 15.05it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 48: train_loss=0.6931490302085876, eval_loss=0.6930854916572571, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25


100%|██████████| 13/13 [00:00<00:00, 15.34it/s]

Epoch 49: train_loss=0.6932047009468079, eval_loss=0.6930854916572571, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# After training, output the predictions
# Convert predictions and true labels to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': eval_preds
})
print(results_df)

    true_label  predicted_label
0            0                1
1            1                1
2            0                1
3            1                1
4            0                1
..         ...              ...
95           0                1
96           0                1
97           1                1
98           1                1
99           1                1

[100 rows x 2 columns]


In [ ]:
# Save the results to a CSV file
results_df.to_csv("/content/drive/MyDrive/model_predictions.csv", index=False)
print("Predictions saved to /content/drive/MyDrive/model_predictions.csv")

# Or print the first few rows of the results
print(results_df.head())

Predictions saved to /content/drive/MyDrive/model_predictions.csv
   true_label  predicted_label
0           0                1
1           1                1
2           0                1
3           1                1
4           0                1


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
accuracy = accuracy_score(true_labels, eval_preds)
f1 = f1_score(true_labels, eval_preds, average='weighted')
recall = recall_score(true_labels, eval_preds, average='weighted')
precision = precision_score(true_labels, eval_preds, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.5
F1 Score: 0.33333333333333326
Recall: 0.5
Precision: 0.25


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
